# ASR 모델 테스트 — Apple Silicon (macOS M4 Pro)

`lab/ASR-model-test/test-way.md`(ASR 평가 프로토콜)과 `ASR-test-lab.md`를 따릅니다.

Colab/T4 전제 엔진(faster-whisper 등) 대신 **Apple Silicon 최적화 프레임워크**(`mlx-whisper`, torch MPS)로 교체했고,
엔진 교체는 **어댑터 패턴**(test-way.md §3.10)으로 구현되어 있습니다.

- **테스트 트랙** (test-way.md §4.1): clean / noisy(SNR 5dB) / code-switched / silence / streaming
- **지표 모듈**: `asr_metrics.py` (CER·WER, RTF·지연, 신뢰도·환각, 노이즈 주입, 벤치마크 하네스)
- **어댑터 모듈**: `asr_adapters.py` (계약 dict + 팩토리)
- **유틸 모듈**: `asr_utils.py` (환경 체크, 오디오 I/O, TTS 평가 세트, 스트리밍 시뮬레이션)

## 0. 환경 확인

In [ ]:
import json          # 결과 JSON 저장용
import logging       # 외부 라이브러리 로그(잡음) 제어용
import os
import sys
from pathlib import Path

# HF 허브 / HTTP 라이브러리의 INFO 로그를 꺼서 노트북 출력을 깔끔하게 유지.
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("urllib3").setLevel(logging.ERROR)

# ── import 경로 부트스트랩 ──────────────────────────────────────
# 노트북을 어느 디렉터리에서 실행하든(랩 폴더 or 저장소 루트) asr_* 모듈을
# import할 수 있도록 파이썬 경로에 해당 폴더를 추가한다.
NB_DIR = Path.cwd()
if (NB_DIR / "asr_adapters.py").exists():
    sys.path.insert(0, str(NB_DIR))                        # 랩 폴더에서 실행
else:
    sys.path.insert(0, str(NB_DIR / "lab" / "ASR-model-test"))  # 저장소 루트에서 실행
# 실험 데이터(wav)를 저장할 저장소 루트 결정. app.py가 있으면 그 폴더가 루트.
REPO_ROOT = NB_DIR if (NB_DIR / "app.py").exists() else NB_DIR.parent.parent

import numpy as np
import pandas as pd

# 이 랩의 세 모듈에서 필요한 함수/상수만 가져온다.
from asr_utils import (env_check_mac, build_eval_set, summarize_eval_set,
                       load_audio, simulate_stream, save_results_json,
                       DATASETS, SAMPLE_RATE)
from asr_metrics import (cer, speech_ratio, is_hallucination,
                         hallucination_rate, streaming_metrics,
                         run_benchmark, quantization_gap)
from asr_adapters import (create_adapter, adapter_available,
                          make_transcribe_fn)

# 환경 점검(파이썬/MLX/MPS/엔진/패키지) → 어떤 엔진을 쓸 수 있는지 확인.
env_check_mac()
print(f"\n샘플레이트: {SAMPLE_RATE} Hz")

## 1. 테스트 트랙 데이터 생성 (gTTS)

clean · code-switched 는 gTTS(한국어 TTS)로 합성하고, noisy 는 clean 에 SNR 5dB 백색소음을 주입합니다
(`asr_metrics.add_noise`). silence 는 순수 침묵(0dBFS)과 미세 소음(≈-60dBFS) 2종입니다.
생성된 wav는 `data/temp_audio/`(gitignore 대상)에 저장됩니다.

In [ ]:
# ── 실험 설정 ──────────────────────────────────────────────────
DATASET_ID = "v2"   # ← 실험 사이클별 데이터셋 (v1 / v2) 선택
DATA_DIR = REPO_ROOT / "data" / "temp_audio" / "asr_test" / DATASET_ID
RESULT_DIR = NB_DIR / "results"

# test-way.md 4.1의 5개 트랙(clean/noisy/code_switched/silence/streaming)을
# 포함한 평가 세트를 gTTS 합성으로 생성한다. 생성 wav는 data/에 저장(무시됨).
eval_set = build_eval_set(DATASETS[DATASET_ID], DATA_DIR)
print(f"[{DATASET_ID}] 평가 세트 {len(eval_set)}건 → {DATA_DIR}")
summarize_eval_set(eval_set)

## 2. 어댑터 팩토리 + 스모크 테스트

사용 가능한 엔진만 팩토리로 생성합니다. 미설치 엔진(sensevoice / qwen3-asr)은 건너뜁니다.
각 어댑터는 `transcribe(audio) -> 계약 dict` (test-way.md §3.10)를 반환합니다.

In [ ]:
# ── 어댑터 팩토리 + 스모크 테스트 ──────────────────────────────
# 평가할 엔진 목록 (qwen3-asr은 transformers 버전 충돌로 미설치 시 자동 제외).
ENGINES = ["mlx-whisper", "openai-whisper", "sensevoice", "qwen3-asr"]
# 설치돼 있어 실제로 사용할 수 있는 엔진만 골라낸다 (어댑터 패턴의 가드).
ENABLED = {e: adapter_available(e) for e in ENGINES}
print("엔진 가용성:", "  ".join(f"{k} {'✅' if v else '❌'}" for k, v in ENABLED.items()))

# 사용 가능한 엔진만 팩토리로 인스턴스 생성 (모델 로드는 첫 호출 때 지연 실행).
adapters = {e: create_adapter(e) for e in ENGINES if ENABLED[e]}
print("\n[스모크 테스트] clean 1건")
# 각 엔진이 clean 1건을 정상 전사하는지 확인 (모델 다운로드/로드도 여기서 발생).
for e, a in adapters.items():
    r = a.transcribe(eval_set[0][0], utt_id="smoke")
    print(f"  [{e}] {r['text'][:38]!r}  conf={r['confidence_ok']}  "
          f"avg_logprob={r['avg_logprob']}")

## 3. 비교 매트릭스 (test-way.md §4.3)

각 엔진을 `run_benchmark` 하네스에 태워 조건별 CER · 평균 지연 · RTF를 계산합니다.

In [ ]:
# ── 비교 매트릭스 (test-way.md 4.3) ────────────────────────────
matrix_rows = []
for e, a in adapters.items():
    # run_benchmark: 엔진 전체를 18건 평가 세트에 돌려
    # 조건별 CER + 평균 지연/RTF 한 줄을 만든다 (비교 매트릭스의 한 행).
    row = run_benchmark(e, make_transcribe_fn(a), eval_set)
    row.pop("n", None)   # 항목 수는 표에서 제외
    matrix_rows.append(row)

# 행 = 엔진, 열 = 지표로 표 구성.
matrix_df = pd.DataFrame(matrix_rows).set_index("engine")
pd.set_option("display.float_format", lambda v: f"{v:.3f}")
matrix_df

## 4. 침묵 트랙 → 환각 방어 (P1)

모든 엔진이 침묵에서 환각을 낼 수 있으므로(test-way.md §3.2), VAD 게이트에 더해 텍스트 레벨 3신호 필터를 적용합니다.
아래는 엔진별 환각률(`hallucination_rate`)과 실제 환각 텍스트 예시입니다.

In [ ]:
# ── 침묵 트랙 → 환각 방어 (P1) ─────────────────────────────────
def silence_analysis(adapter):
    """침묵 트랙 전부를 전사해 (텍스트, 발화비율, 환각 여부) 기록을 만든다."""
    recs = []
    for path, ref, cond, dur in eval_set:
        if cond != "silence":
            continue
        y = load_audio(path)
        r = adapter.transcribe(path, utt_id=Path(path).stem)
        ratio = speech_ratio(y, SAMPLE_RATE)   # RMS 기반 발화 비율 (침묵이면 ≈0)
        # 3신호 필터(침묵 역설 / 정형구 / 토큰 반복)로 환각 여부 판정 (asr_metrics).
        recs.append({"utt_id": r["utt_id"], "text": r["text"],
                     "speech_ratio": round(ratio, 3),
                     "hallucination": is_hallucination(r["text"], dur, ratio)})
    return recs

hallu_rows = []
for e, a in adapters.items():
    recs = silence_analysis(a)
    # 환각률 = 환각 판정 발화 수 / 전체 침묵 발화 수 (목표: 0%).
    rate = hallucination_rate([r["text"] for r in recs],
                             [r["speech_ratio"] for r in recs])
    hallu_rows.append({"engine": e, "hallucination_rate": rate})
    for r in recs:
        r["engine"] = e
    print(f"[{e}] 환각률 {rate:.0%}",
          " | 환각 텍스트:", [r['text'] for r in recs if r['hallucination']] or "없음")

hallucination_df = pd.DataFrame(hallu_rows).set_index("engine")
hallucination_df

## 5. 2-Tier 신뢰도 전략 (P2)

greedy 디코딩 → `avg_logprob < -1.0` 이면 beam=5 로 1회 재시도 (test-way.md §3.3).

In [ ]:
# ── 2-Tier 신뢰도 전략 (P2, test-way.md 3.3) ───────────────────
if "mlx-whisper" in adapters:
    mlx = adapters["mlx-whisper"]
    noisy_items = [x for x in eval_set if x[2] == "noisy"]  # noisy 트랙만 추림

    # 가장 신뢰도(avg_logprob)가 낮은 noisy 발화를 1차로 골라낸다.
    worst_item, worst_lp = None, float("inf")
    for it in noisy_items:
        lp = mlx.transcribe(it[0])["avg_logprob"] or -99.0
        if lp < worst_lp:
            worst_item, worst_lp = it, lp

    print(f"최저 신뢰도 noisy 발화: {Path(worst_item[0]).name}  avg_logprob={worst_lp:.3f}")
    # 1단계: greedy 디코딩 (빠름, 실시간 기본값).
    greedy = mlx.transcribe(worst_item[0], utt_id="tier1_greedy")
    print(f"  greedy : {greedy['text']}  conf={greedy['confidence_ok']}")

    # 2단계: avg_logprob < -1.0 (신뢰도 불충분)일 때만 beam=5로 1회 재시도.
    retry = greedy
    if not greedy["confidence_ok"]:
        mlx_beam = create_adapter("mlx-whisper", beam_size=5)
        retry = mlx_beam.transcribe(worst_item[0], utt_id="tier2_beam")
        print(f"  beam=5 : {retry['text']}  conf={retry['confidence_ok']}")

    print(f"  CER  greedy={cer(worst_item[1], greedy['text']):.1%} → beam={cer(worst_item[1], retry['text']):.1%}")
else:
    print("mlx-whisper 없음 — 건너뜀")

## 6. 양자화 허용 오차 (test-way.md §3.6)

`mlx-community/whisper-large-v3-turbo-q4`(4bit)와 fp16의 CER 차이를 **측정**합니다.
"측정하라, 가정하지 마라" — test-way.md §3.6.

In [ ]:
# ── 양자화 허용 오차 (test-way.md 3.6) ─────────────────────────
if "mlx-whisper" in adapters:
    # 4bit 양자화 모델 (fp16 ~1.6GB → q4 ~0.8GB, 메모리 절반).
    q4 = create_adapter("mlx-whisper",
                        model_id="mlx-community/whisper-large-v3-turbo-q4")
    items = [x for x in eval_set if x[2] in ("clean", "noisy")]
    # mlx-whisper는 모델 1개만 캐시하므로, 모델별로 묶어 전사해
    # fp16↔q4 재로드를 최소화한다.
    hyps = {"fp16": {}, "q4": {}}
    for key, ad in (("fp16", mlx), ("q4", q4)):
        for path, ref, cond, dur in items:
            hyps[key][Path(path).stem] = ad.transcribe(path)["text"]

    # fp16 대비 q4의 CER 차이를 항목별로 비교. "측정하라, 가정하지 마라".
    quant_rows = []
    print(f"{'item':<16}{'cond':<8}{'fp16 CER':>10}{'q4 CER':>10}{'Δ':>10}")
    for path, ref, cond, dur in items:
        stem = Path(path).stem
        c_fp = cer(ref, hyps["fp16"][stem])
        c_q4 = cer(ref, hyps["q4"][stem])
        gap = quantization_gap(c_fp, c_q4)   # 절대/상대 차이 계산
        quant_rows.append({"item": stem, "cond": cond,
                           "fp16_cer": c_fp, "q4_cer": c_q4,
                           "gap_abs": gap["absolute"]})
        print(f"{stem:<16}{cond:<8}{c_fp:>10.1%}{c_q4:>10.1%}{gap['absolute']:>10.1%}")
    quant_df = pd.DataFrame(quant_rows)
    print("\n평균 CER 차이(q4−fp16):",
          f"{quant_df['gap_abs'].mean():+.2%}")

## 7. initial_prompt 도메인 주입 (P5 대안)

한국어 파인튜닝 대신 무비용으로 도메인 컨텍스트를 주입해 코드 스위칭 발화의 CER을 개선합니다
(test-way.md §3.5, §3.9).

In [ ]:
# ── initial_prompt 도메인 주입 (P5 대안, test-way.md 3.5/3.9) ──
if "mlx-whisper" in adapters:
    # 강의 도메인에 자주 등장하는 영어 용어를 프롬프트로 주입 (파인튜닝 대체).
    prompt = "API, response, timeout, tokenizer, embedding, gradient, memory, database"
    prompted = create_adapter("mlx-whisper", initial_prompt=prompt)
    cs_items = [x for x in eval_set if x[2] == "code_switched"]
    print(f"initial_prompt: {prompt!r}\n")
    print(f"{'item':<14}{'base CER':>10}{'prompt CER':>12}")
    prompt_rows = []
    # 주입 전후의 CER을 항목별로 비교한다.
    for path, ref, cond, dur in cs_items:
        t_base = mlx.transcribe(path)["text"]          # 주입 없음 (기본)
        t_prompt = prompted.transcribe(path)["text"]   # 프롬프트 주입
        c_base, c_prompt = cer(ref, t_base), cer(ref, t_prompt)
        prompt_rows.append({"item": Path(path).stem, "base_cer": c_base,
                           "prompt_cer": c_prompt})
        print(f"{Path(path).stem:<14}{c_base:>10.1%}{c_prompt:>12.1%}")
    prompt_df = pd.DataFrame(prompt_rows)

## 8. 스트리밍 시뮬레이션 (P6)

성장 버퍼 재전사 방식으로 실시간 스트리밍을 시뮬레이션하고
`first_partial_ms` / `final_latency_ms` / RTF를 측정합니다 (test-way.md §2.2, §3.8).

In [ ]:
# ── 스트리밍 시뮬레이션 (P6, test-way.md 3.8) ──────────────────
if "mlx-whisper" in adapters:
    clean_item = next(x for x in eval_set if x[2] == "clean")  # 발화 1개 선택
    y = load_audio(clean_item[0])
    # 1s 청크로 성장 버퍼를 재전사하는 시뮬레이션.
    # 반환: partial/final 이벤트 목록, 발화 길이, 벽시계 시간.
    events, speech_ms, wall_s = simulate_stream(y, mlx, chunk_s=1.0,
                                                eou_silence_chunks=2)
    # first_partial_ms / final_latency_ms / RTF 지표 계산.
    stream_metrics = streaming_metrics(events, speech_ms, wall_s)
    print(f"오디오 길이: {len(y)/SAMPLE_RATE:.1f}s  발화 판정: {speech_ms:.0f}ms  "
          f"벽시계: {wall_s:.1f}s")
    print(f"streaming_metrics: {stream_metrics}")
    display(pd.DataFrame(events))   # 이벤트 스트림을 표로 확인
else:
    print("mlx-whisper 없음 — 건너뜀")

## 9. 결과 저장

이번 사이클의 지표를 JSON으로 저장해 두 사이클 종합 결과(마크다운) 작성에 사용합니다.

In [ ]:
# ── 결과 저장 ──────────────────────────────────────────────────
# 이번 사이클의 지표를 JSON으로 저장 (후속 마크다운 결과 정리에 사용).
report = {
    "dataset_id": DATASET_ID,          # 데이터셋 식별자 (v1/v2)
    "enabled_engines": list(adapters), # 실제 실행된 엔진 목록
    "matrix": matrix_rows,             # 비교 매트릭스 행 (조건별 CER 등)
    "hallucination": hallu_rows,       # 침묵 트랙 환각률
    "streaming": stream_metrics if "stream_metrics" in dir() else None,
}
save_results_json(report, RESULT_DIR / f"asr_results_{DATASET_ID}.json")
print(f"저장 완료 → {RESULT_DIR / f'asr_results_{DATASET_ID}.json'}")